# 00 — Setup: SQL Server

Cria o banco de dados `seguradora` no SQL Server e importa os dados dos arquivos CSV da pasta `data/`.

In [ ]:
import os
import csv
import pyodbc
from dotenv import load_dotenv

load_dotenv()

## Conexão

In [ ]:
conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={os.getenv('SQLSERVER_HOST')},{os.getenv('SQLSERVER_PORT')};"
    f"UID={os.getenv('SQLSERVER_USER')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str, autocommit=True)
cursor = conn.cursor()
print("Conectado ao SQL Server!")

## Criação do banco e tabelas

In [ ]:
DB = os.getenv("SQLSERVER_DB")

cursor.execute(f"IF DB_ID('{DB}') IS NOT NULL DROP DATABASE {DB}")
cursor.execute(f"CREATE DATABASE {DB}")
cursor.execute(f"USE {DB}")
print(f"Banco '{DB}' criado.")

ddl = """
CREATE TABLE estado (
    id_estado INT PRIMARY KEY,
    sigla CHAR(2),
    nome NVARCHAR(50)
);

CREATE TABLE regiao (
    id_regiao INT PRIMARY KEY,
    nome NVARCHAR(50)
);

CREATE TABLE municipio (
    id_municipio INT PRIMARY KEY,
    nome NVARCHAR(100),
    id_estado INT REFERENCES estado(id_estado)
);

CREATE TABLE endereco (
    id_endereco INT PRIMARY KEY,
    logradouro NVARCHAR(150),
    numero NVARCHAR(10),
    bairro NVARCHAR(100),
    cep NVARCHAR(10),
    id_municipio INT REFERENCES municipio(id_municipio)
);

CREATE TABLE cliente (
    id_cliente INT PRIMARY KEY,
    nome NVARCHAR(100),
    cpf NVARCHAR(14),
    data_nascimento DATE,
    email NVARCHAR(150),
    id_endereco INT REFERENCES endereco(id_endereco)
);

CREATE TABLE telefone (
    id_telefone INT PRIMARY KEY,
    numero NVARCHAR(20),
    tipo NVARCHAR(20),
    id_cliente INT REFERENCES cliente(id_cliente)
);

CREATE TABLE marca (
    id_marca INT PRIMARY KEY,
    nome NVARCHAR(50),
    pais_origem NVARCHAR(50)
);

CREATE TABLE modelo (
    id_modelo INT PRIMARY KEY,
    nome NVARCHAR(50),
    id_marca INT REFERENCES marca(id_marca),
    tipo NVARCHAR(30)
);

CREATE TABLE carro (
    id_carro INT PRIMARY KEY,
    placa NVARCHAR(10),
    ano INT,
    cor NVARCHAR(30),
    id_modelo INT REFERENCES modelo(id_modelo),
    id_cliente INT REFERENCES cliente(id_cliente)
);

CREATE TABLE apolice (
    id_apolice INT PRIMARY KEY,
    numero NVARCHAR(20),
    data_inicio DATE,
    data_fim DATE,
    valor_premio DECIMAL(10,2),
    cobertura NVARCHAR(30),
    id_cliente INT REFERENCES cliente(id_cliente),
    id_carro INT REFERENCES carro(id_carro)
);

CREATE TABLE sinistro (
    id_sinistro INT PRIMARY KEY,
    data_ocorrencia DATE,
    descricao NVARCHAR(255),
    valor_prejuizo DECIMAL(10,2),
    status NVARCHAR(30),
    id_apolice INT REFERENCES apolice(id_apolice)
);
"""

for stmt in ddl.strip().split(";"):
    stmt = stmt.strip()
    if stmt:
        cursor.execute(stmt)

print("Tabelas criadas!")

## Importação dos CSVs

In [ ]:
DATA_DIR = "../data"

TABLES = [
    "estado", "regiao", "municipio", "endereco",
    "cliente", "telefone", "marca", "modelo",
    "carro", "apolice", "sinistro"
]

for table in TABLES:
    path = os.path.join(DATA_DIR, f"{table}.csv")
    with open(path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        if not rows:
            continue
        cols = ", ".join(rows[0].keys())
        placeholders = ", ".join(["?"] * len(rows[0]))
        sql = f"INSERT INTO {table} ({cols}) VALUES ({placeholders})"
        for row in rows:
            cursor.execute(sql, list(row.values()))
    conn.commit()
    print(f"  {table}: {len(rows)} registros inseridos")

print("\nImportação concluída.")

## Verificação

In [ ]:
for table in TABLES:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  {table}: {count} registros")

cursor.close()
conn.close()